In [1]:
# ========================================================================
# ETL DATA WAREHOUSE - RENT4YOU
# Caso de Estudio #2: Preprocesamiento de Datos
# ========================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import os
warnings.filterwarnings('ignore')

print("=" * 70)
print("ETL DATA WAREHOUSE - RENT4YOU")
print("Preprocesamiento de Datos para Análisis de Ventas por Sucursal")
print("=" * 70)

# ========================================================================
# 1. GENERACIÓN DE DATOS CRUDOS (Simulación de bases operacionales)
# ========================================================================

print("\n" + "=" * 70)
print("FASE 1: GENERACIÓN DE DATOS CRUDOS")
print("=" * 70)

# 1.1 BASE DE DATOS: SUCURSALES (Datos maestros)
print("\n[1/5] Generando datos de Sucursales...")
sucursales_raw = pd.DataFrame({
    'id_sucursal': [1, 2, 3, 4, 5, 5, 6],  # Duplicado intencional (id 5)
    'nombre_sucursal': ['Centro CDMX', 'Norte MTY', 'Sur GDL', 'Este Puebla', None, 'Occidente QRO', 'Bajío León'],
    'ciudad': ['CDMX', 'monterrey', 'Guadalajara', 'Puebla', 'Querétaro', 'Querétaro', '  León  '],
    'region': ['Centro', 'Norte', 'Occidente', 'Centro', 'Centro', 'Centro', 'Bajío']
})
print(f"   ✓ Generados {len(sucursales_raw)} registros (con duplicados y nulos)")

# 1.2 BASE DE DATOS: CLIENTES (Datos maestros)
print("\n[2/5] Generando datos de Clientes...")
np.random.seed(42)
nombres = ['Ana García', 'Carlos López', 'María Rodríguez', 'José Martínez', 'Laura Hernández',
           'Miguel Pérez', 'Carmen González', 'Francisco Sánchez', 'Isabel Ramírez', 'Antonio Torres']

clientes_raw = pd.DataFrame({
    'id_cliente': list(range(1, 102)) + [50],  # Duplicado intencional (id 50)
    'nombre': np.random.choice(nombres, 102),
    'edad': np.random.randint(18, 75, 102),
    'ciudad': np.random.choice(['CDMX', 'Monterrey', 'Guadalajara', 'Puebla', 'Querétaro', None], 102),
    'genero': np.random.choice(['M', 'F', 'Otro', None], 102),
    'tipo_cliente': np.random.choice(['Regular', 'VIP', 'Corporativo', 'Regular'], 102)
})
# Introducir más valores nulos intencionalmente
clientes_raw.loc[0, 'ciudad'] = None
clientes_raw.loc[25, 'genero'] = None
clientes_raw.loc[75, 'ciudad'] = None
print(f"   ✓ Generados {len(clientes_raw)} registros (con duplicados y nulos)")

# 1.3 BASE DE DATOS: VEHÍCULOS (Catálogo)
print("\n[3/5] Generando catálogo de Vehículos...")
vehiculos_raw = pd.DataFrame({
    'id_vehiculo': range(1, 21),
    'tipo_vehiculo': np.random.choice(['Auto', 'Moto', 'Camioneta', 'Remolque', 'SUV'], 20),
    'marca': np.random.choice(['Toyota', 'Honda', 'Ford', 'Chevrolet', 'Nissan', 'Mazda'], 20),
    'modelo': [f'Modelo {chr(65+i)}' for i in range(20)],
    'ano_fabricacion': np.random.randint(2015, 2025, 20),
    'costo_diario': np.random.uniform(300, 2500, 20).round(2)
})
# Introducir valores problemáticos
vehiculos_raw.loc[0, 'costo_diario'] = -500.00  # Costo negativo (error)
vehiculos_raw.loc[5, 'costo_diario'] = 0  # Costo cero (error)
vehiculos_raw.loc[10, 'ano_fabricacion'] = 2030  # Año futuro (inconsistencia)
print(f"   ✓ Generados {len(vehiculos_raw)} registros (con valores inválidos)")

# 1.4 DIMENSIÓN TIEMPO (Generación automática)
print("\n[4/5] Generando Dimensión Tiempo...")
fecha_inicio = datetime(2023, 1, 1)
fecha_fin = datetime(2024, 12, 31)
fechas = pd.date_range(fecha_inicio, fecha_fin, freq='D')

tiempo_raw = pd.DataFrame({
    'id_fecha': range(1, len(fechas) + 1),
    'fecha': fechas,
    'mes': fechas.month,
    'trimestre': fechas.quarter,
    'ano': fechas.year,
    'dia_semana': fechas.day_name()
})
print(f"   ✓ Generados {len(tiempo_raw)} registros (calendario completo 2023-2024)")

# 1.5 BASE DE DATOS: VENTAS (Transaccional)
print("\n[5/5] Generando datos de Ventas...")
np.random.seed(123)
n_ventas = 500

ventas_raw = pd.DataFrame({
    'id_venta': range(1, n_ventas + 1),
    'id_cliente': np.random.randint(1, 102, n_ventas),
    'id_vehiculo': np.random.randint(1, 21, n_ventas),
    'id_sucursal': np.random.randint(1, 7, n_ventas),
    'id_fecha': np.random.randint(1, len(fechas) + 1, n_ventas),
    'cantidad_dias': np.random.randint(1, 31, n_ventas),
    'monto_total': np.random.uniform(500, 20000, n_ventas).round(2),
    'descuento': np.random.uniform(0, 1000, n_ventas).round(2),
    'metodo_pago': np.random.choice(['Efectivo', 'Tarjeta', 'Transferencia'], n_ventas)
})

# Introducir problemas de calidad
ventas_raw = pd.concat([ventas_raw, ventas_raw.iloc[[0, 1, 2]]], ignore_index=True)  # Duplicados
ventas_raw.loc[10, 'cantidad_dias'] = -5  # Valor negativo
ventas_raw.loc[20, 'monto_total'] = -1000  # Monto negativo
ventas_raw.loc[30, 'id_cliente'] = 999  # Referencia inválida
ventas_raw.loc[40, 'id_vehiculo'] = 999  # Referencia inválida
ventas_raw.loc[50, 'id_sucursal'] = 999  # Referencia inválida

print(f"   ✓ Generados {len(ventas_raw)} registros (con duplicados y referencias inválidas)")

print("\n✓ Datos crudos generados exitosamente")
print(f"\nTotal de registros crudos: {len(sucursales_raw) + len(clientes_raw) + len(vehiculos_raw) + len(tiempo_raw) + len(ventas_raw)}")

# ========================================================================
# 2. PROCESO ETL: EXTRACCIÓN, TRANSFORMACIÓN Y LIMPIEZA
# ========================================================================

print("\n" + "=" * 70)
print("FASE 2: PROCESO ETL - LIMPIEZA Y TRANSFORMACIÓN")
print("=" * 70)

# 2.1 LIMPIEZA: DIM_SUCURSAL
print("\n[1/5] Limpiando DIM_SUCURSAL...")
dim_sucursal = sucursales_raw.copy()

# Técnica 1: Eliminación de duplicados
registros_antes = len(dim_sucursal)
dim_sucursal = dim_sucursal.drop_duplicates(subset=['id_sucursal'], keep='first')
duplicados_eliminados = registros_antes - len(dim_sucursal)
print(f"   ✓ Duplicados eliminados: {duplicados_eliminados}")

# Técnica 2: Manejo de valores faltantes
nulos_antes = dim_sucursal['nombre_sucursal'].isnull().sum()
dim_sucursal['nombre_sucursal'] = dim_sucursal['nombre_sucursal'].fillna('No especificado')
print(f"   ✓ Valores nulos manejados: {nulos_antes}")

# Técnica 3: Normalización de texto
dim_sucursal['ciudad'] = dim_sucursal['ciudad'].str.strip().str.title()
dim_sucursal['region'] = dim_sucursal['region'].str.strip().str.title()
dim_sucursal['nombre_sucursal'] = dim_sucursal['nombre_sucursal'].str.strip()
print(f"   ✓ Normalización de texto completada")

print(f"   → Registros finales: {len(dim_sucursal)}")

# 2.2 LIMPIEZA: DIM_CLIENTE
print("\n[2/5] Limpiando DIM_CLIENTE...")
dim_cliente = clientes_raw.copy()

# Técnica 1: Eliminación de duplicados
registros_antes = len(dim_cliente)
dim_cliente = dim_cliente.drop_duplicates(subset=['id_cliente'], keep='first')
duplicados_eliminados = registros_antes - len(dim_cliente)
print(f"   ✓ Duplicados eliminados: {duplicados_eliminados}")

# Técnica 2: Manejo de valores faltantes
nulos_ciudad = dim_cliente['ciudad'].isnull().sum()
nulos_genero = dim_cliente['genero'].isnull().sum()
dim_cliente['ciudad'] = dim_cliente['ciudad'].fillna('No especificado')
dim_cliente['genero'] = dim_cliente['genero'].fillna('No especificado')
print(f"   ✓ Valores nulos manejados: {nulos_ciudad + nulos_genero}")

# Técnica 3: Validación de rangos
registros_antes = len(dim_cliente)
dim_cliente = dim_cliente[(dim_cliente['edad'] >= 18) & (dim_cliente['edad'] <= 100)]
invalidos = registros_antes - len(dim_cliente)
print(f"   ✓ Validación de edad (18-100 años): {invalidos} registros inválidos eliminados")

# Técnica 4: Normalización
dim_cliente['ciudad'] = dim_cliente['ciudad'].str.strip().str.title()
dim_cliente['nombre'] = dim_cliente['nombre'].str.strip().str.title()
print(f"   ✓ Normalización de texto completada")

print(f"   → Registros finales: {len(dim_cliente)}")

# 2.3 LIMPIEZA: DIM_VEHICULO
print("\n[3/5] Limpiando DIM_VEHICULO...")
dim_vehiculo = vehiculos_raw.copy()

# Técnica 1: Validación de rangos (costos)
registros_antes = len(dim_vehiculo)
dim_vehiculo = dim_vehiculo[dim_vehiculo['costo_diario'] > 0]
invalidos = registros_antes - len(dim_vehiculo)
print(f"   ✓ Costos negativos/cero eliminados: {invalidos}")

# Técnica 2: Validación de rangos (años)
registros_antes = len(dim_vehiculo)
ano_actual = datetime.now().year
dim_vehiculo = dim_vehiculo[(dim_vehiculo['ano_fabricacion'] >= 2000) & 
                             (dim_vehiculo['ano_fabricacion'] <= ano_actual)]
invalidos = registros_antes - len(dim_vehiculo)
print(f"   ✓ Años de fabricación inválidos eliminados: {invalidos}")

# Técnica 3: Normalización
dim_vehiculo['tipo_vehiculo'] = dim_vehiculo['tipo_vehiculo'].str.strip().str.title()
dim_vehiculo['marca'] = dim_vehiculo['marca'].str.strip().str.title()
dim_vehiculo['modelo'] = dim_vehiculo['modelo'].str.strip()
print(f"   ✓ Normalización de texto completada")

print(f"   → Registros finales: {len(dim_vehiculo)}")

# 2.4 LIMPIEZA: DIM_TIEMPO
print("\n[4/5] Validando DIM_TIEMPO...")
dim_tiempo = tiempo_raw.copy()

# Técnica 1: Validación de consistencia de fechas
dim_tiempo['fecha'] = pd.to_datetime(dim_tiempo['fecha'])
print(f"   ✓ Formato de fechas validado")

# Verificar que no haya fechas duplicadas
duplicados = dim_tiempo.duplicated(subset=['fecha']).sum()
print(f"   ✓ Fechas duplicadas: {duplicados}")

print(f"   → Registros finales: {len(dim_tiempo)}")

# 2.5 LIMPIEZA: FACT_VENTAS
print("\n[5/5] Limpiando FACT_VENTAS...")
fact_ventas = ventas_raw.copy()

# Técnica 1: Eliminación de duplicados
registros_antes = len(fact_ventas)
fact_ventas = fact_ventas.drop_duplicates()
duplicados_eliminados = registros_antes - len(fact_ventas)
print(f"   ✓ Duplicados eliminados: {duplicados_eliminados}")

# Técnica 2: Validación de rangos
registros_antes = len(fact_ventas)
fact_ventas = fact_ventas[fact_ventas['cantidad_dias'] > 0]
fact_ventas = fact_ventas[fact_ventas['monto_total'] > 0]
invalidos = registros_antes - len(fact_ventas)
print(f"   ✓ Valores negativos/cero eliminados: {invalidos}")

# Técnica 3: Validación de integridad referencial
registros_antes = len(fact_ventas)

# Validar id_cliente
fact_ventas = fact_ventas[fact_ventas['id_cliente'].isin(dim_cliente['id_cliente'])]
invalidos_cliente = registros_antes - len(fact_ventas)

# Validar id_vehiculo
registros_antes = len(fact_ventas)
fact_ventas = fact_ventas[fact_ventas['id_vehiculo'].isin(dim_vehiculo['id_vehiculo'])]
invalidos_vehiculo = registros_antes - len(fact_ventas)

# Validar id_sucursal
registros_antes = len(fact_ventas)
fact_ventas = fact_ventas[fact_ventas['id_sucursal'].isin(dim_sucursal['id_sucursal'])]
invalidos_sucursal = registros_antes - len(fact_ventas)

# Validar id_fecha
registros_antes = len(fact_ventas)
fact_ventas = fact_ventas[fact_ventas['id_fecha'].isin(dim_tiempo['id_fecha'])]
invalidos_fecha = registros_antes - len(fact_ventas)

total_invalidos = invalidos_cliente + invalidos_vehiculo + invalidos_sucursal + invalidos_fecha
print(f"   ✓ Integridad referencial validada:")
print(f"      - Referencias inválidas a clientes: {invalidos_cliente}")
print(f"      - Referencias inválidas a vehículos: {invalidos_vehiculo}")
print(f"      - Referencias inválidas a sucursales: {invalidos_sucursal}")
print(f"      - Referencias inválidas a fechas: {invalidos_fecha}")
print(f"      - Total eliminados: {total_invalidos}")

print(f"   → Registros finales: {len(fact_ventas)}")

print("\n✓ Proceso ETL completado exitosamente")

# ========================================================================
# 3. RESUMEN Y ESTADÍSTICAS DE LIMPIEZA
# ========================================================================

print("\n" + "=" * 70)
print("FASE 3: RESUMEN DE DATOS PREPROCESADOS")
print("=" * 70)

print("\n📊 TABLAS DEL DATA WAREHOUSE:")
print(f"   1. DIM_SUCURSAL:    {len(dim_sucursal):>5} registros")
print(f"   2. DIM_CLIENTE:     {len(dim_cliente):>5} registros")
print(f"   3. DIM_VEHICULO:    {len(dim_vehiculo):>5} registros")
print(f"   4. DIM_TIEMPO:      {len(dim_tiempo):>5} registros")
print(f"   5. FACT_VENTAS:     {len(fact_ventas):>5} registros")
print(f"   {'─' * 40}")
print(f"   TOTAL:              {len(dim_sucursal) + len(dim_cliente) + len(dim_vehiculo) + len(dim_tiempo) + len(fact_ventas):>5} registros")

# ========================================================================
# 4. ANÁLISIS EXPLORATORIO BÁSICO
# ========================================================================

print("\n" + "=" * 70)
print("FASE 4: ANÁLISIS EXPLORATORIO")
print("=" * 70)

# 4.1 Análisis de Ventas por Sucursal
print("\n📈 VENTAS POR SUCURSAL:")
ventas_por_sucursal = fact_ventas.merge(dim_sucursal, on='id_sucursal')
resumen_sucursal = ventas_por_sucursal.groupby(['nombre_sucursal', 'ciudad']).agg({
    'monto_total': ['sum', 'mean', 'count']
}).round(2)
resumen_sucursal.columns = ['Monto Total', 'Ticket Promedio', 'Num. Rentas']
print(resumen_sucursal)

# 4.2 Análisis de Vehículos más Rentados
print("\n🚗 VEHÍCULOS MÁS RENTADOS:")
vehiculos_rentados = fact_ventas.merge(dim_vehiculo, on='id_vehiculo')
top_vehiculos = vehiculos_rentados.groupby('tipo_vehiculo').agg({
    'id_venta': 'count',
    'monto_total': 'sum'
}).round(2)
top_vehiculos.columns = ['Cantidad Rentas', 'Ingresos Totales']
top_vehiculos = top_vehiculos.sort_values('Cantidad Rentas', ascending=False)
print(top_vehiculos)

# 4.3 Análisis de Perfil de Clientes
print("\n👥 PERFIL DE CLIENTES POR TIPO:")
perfil_clientes = dim_cliente.groupby('tipo_cliente').agg({
    'id_cliente': 'count',
    'edad': 'mean'
}).round(2)
perfil_clientes.columns = ['Cantidad', 'Edad Promedio']
print(perfil_clientes)

# ========================================================================
# 5. EXPORTACIÓN DE DATOS PREPROCESADOS
# ========================================================================

print("\n" + "=" * 70)
print("FASE 5: EXPORTACIÓN DE DATOS")
print("=" * 70)

# Crear carpeta si no existe
carpeta_destino = 'datos_preprocesados'
if not os.path.exists(carpeta_destino):
    os.makedirs(carpeta_destino)
    print(f"\n✓ Carpeta '{carpeta_destino}' creada")

# Exportar a CSV
print("\n💾 Exportando tablas a CSV...")

dim_sucursal.to_csv(f'{carpeta_destino}/dim_sucursal.csv', index=False, encoding='utf-8-sig')
print(f"   ✓ dim_sucursal.csv")

dim_cliente.to_csv(f'{carpeta_destino}/dim_cliente.csv', index=False, encoding='utf-8-sig')
print(f"   ✓ dim_cliente.csv")

dim_vehiculo.to_csv(f'{carpeta_destino}/dim_vehiculo.csv', index=False, encoding='utf-8-sig')
print(f"   ✓ dim_vehiculo.csv")

dim_tiempo.to_csv(f'{carpeta_destino}/dim_tiempo.csv', index=False, encoding='utf-8-sig')
print(f"   ✓ dim_tiempo.csv")

fact_ventas.to_csv(f'{carpeta_destino}/fact_ventas.csv', index=False, encoding='utf-8-sig')
print(f"   ✓ fact_ventas.csv")

print(f"\n✓ Todos los archivos exportados exitosamente a '{carpeta_destino}/'")

# ========================================================================
# 6. VALIDACIÓN FINAL
# ========================================================================

print("\n" + "=" * 70)
print("FASE 6: VALIDACIÓN FINAL DE CALIDAD")
print("=" * 70)

print("\n✅ CHECKLIST DE CALIDAD:")
print(f"   ✓ Duplicados eliminados en todas las tablas")
print(f"   ✓ Valores nulos manejados correctamente")
print(f"   ✓ Rangos validados (edades, costos, fechas)")
print(f"   ✓ Integridad referencial verificada")
print(f"   ✓ Normalización de texto aplicada")
print(f"   ✓ Consistencia de datos garantizada")

print("\n" + "=" * 70)
print("✓ PROCESO ETL COMPLETADO EXITOSAMENTE")
print("=" * 70)

print("\n📁 Archivos generados:")
print(f"   → {carpeta_destino}/dim_sucursal.csv")
print(f"   → {carpeta_destino}/dim_cliente.csv")
print(f"   → {carpeta_destino}/dim_vehiculo.csv")
print(f"   → {carpeta_destino}/dim_tiempo.csv")
print(f"   → {carpeta_destino}/fact_ventas.csv")

print("\n🎯 Data Warehouse listo para análisis de Business Intelligence")
print("=" * 70)

ETL DATA WAREHOUSE - RENT4YOU
Preprocesamiento de Datos para Análisis de Ventas por Sucursal

FASE 1: GENERACIÓN DE DATOS CRUDOS

[1/5] Generando datos de Sucursales...
   ✓ Generados 7 registros (con duplicados y nulos)

[2/5] Generando datos de Clientes...
   ✓ Generados 102 registros (con duplicados y nulos)

[3/5] Generando catálogo de Vehículos...
   ✓ Generados 20 registros (con valores inválidos)

[4/5] Generando Dimensión Tiempo...
   ✓ Generados 731 registros (calendario completo 2023-2024)

[5/5] Generando datos de Ventas...
   ✓ Generados 503 registros (con duplicados y referencias inválidas)

✓ Datos crudos generados exitosamente

Total de registros crudos: 1363

FASE 2: PROCESO ETL - LIMPIEZA Y TRANSFORMACIÓN

[1/5] Limpiando DIM_SUCURSAL...
   ✓ Duplicados eliminados: 1
   ✓ Valores nulos manejados: 1
   ✓ Normalización de texto completada
   → Registros finales: 6

[2/5] Limpiando DIM_CLIENTE...
   ✓ Duplicados eliminados: 1
   ✓ Valores nulos manejados: 48
   ✓ Validaci